# Using `data.features` to generate a full dataframe for training

I tried to create an elegant system for designing your own data stack on the fly, but I've failed so far. Instead, use the following method which I will show and then explain.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, "../")

from data import get_data
from data.features import site_df

# hyperparameters
EDGES_M = (50_000, 150_000)
VEL = 1.0
WINDOW = "31D"
CENTER_WINDOW = False

uid = "WQS0039"

# the important bit
df = site_df(uid, edges=EDGES_M, velocity=VEL, roll_window=WINDOW, center_roll=CENTER_WINDOW)

In [2]:
df.columns

Index(['site_uid', 'nitrate_con', 'nitrate_roll', 'nitrate_cal_d',
       'nitrate_cal_w', 'nitrate_cal_m', 'doy_sin', 'doy_cos', 'nitrate_doy',
       'nitrate_woy', 'nitrate_moy', 'date', 'precip_in_1d_b0',
       'precip_in_1d_b1', 'max_temp_b0', 'max_temp_b1', 'min_temp_b0',
       'min_temp_b1', 'max_rel_humidity_b0', 'max_rel_humidity_b1',
       'min_rel_humidity_b0', 'min_rel_humidity_b1', 'vpd_b0', 'vpd_b1',
       'solar_rad_b0', 'solar_rad_b1', 'evapotranspiration_b0',
       'evapotranspiration_b1', 'fuel_moisture_1000h_b0',
       'fuel_moisture_1000h_b1', 'Alfalfa_b0', 'Alfalfa_b1', 'Corn_b0',
       'Corn_b1', 'Fallow_b0', 'Fallow_b1', 'Hay_Pasture_b0', 'Hay_Pasture_b1',
       'Nonag_b0', 'Nonag_b1', 'Other_b0', 'Other_b1', 'Small_Grains_b0',
       'Small_Grains_b1', 'Soybeans_b0', 'Soybeans_b1', 'total_kg_N_b0',
       'total_kg_N_b1', 'surplus_kgha_b0', 'surplus_kgha_b1'],
      dtype='str')

Please skim the code of `site_df` in `data/features.py`, it's at the very end of the file. Here's a description of what it does with quotes from the method.

1. Aggregate weather grid cells into "distance buckets" defined by the `EDGES_M` hyperparameter

```
    cb = agg_crops_by_bucket(uid, edges=edges)
    sb = agg_surplus_by_bucket(uid, edges=edges)
    wb = agg_weather_by_bucket_w_lag(uid, edges=edges, water_velocity=velocity)
```

crop, surplus and weather data are aggregated by distance. In the above setup, cells with distance between $0$ and $50,000$ meters are sent to bucket $0$, cells with distance between $50,000$ and $150,000$ are sent to bucket $1$, and cells with distance above $150,000$ are sent to bucket 2.

*The weather data is lagged based on bucket.* Lags to the rain are added depending on the velocity `VEL` hyperparameter and the bucket the row belongs to. It's lagged roughly by `bucket_dist/velocity`. The VEL hyperparameter should be tuned via cross validation.

2. Build the daily nitrate and store its date index for use

```
    cb = agg_crops_by_bucket(uid, edges=edges)
    sb = agg_surplus_by_bucket(uid, edges=edges)
    wb = agg_weather_by_bucket_w_lag(uid, edges=edges, water_velocity=velocity)
```

3. Add a bunch of seaonality features
```
    # cross-site date-keyed reference features (distinct names so they don't collide)
    n_rolling = nitrate_rolling(window=roll_window, center=center_roll).rename("nitrate_roll")
    n_cal_D = nitrate_avg_calendar(freq="D").rename("nitrate_cal_d")
    n_cal_W = nitrate_avg_calendar(freq="W").rename("nitrate_cal_w")
    n_cal_M = nitrate_avg_calendar(freq="M").rename("nitrate_cal_m")
    pure_signal = doy_climatology_pure_signal(n_daily)  # date-indexed (doy_sin/doy_cos)

    # seasonal nitrate averages mapped onto the calendar dates
    def _help(d, name):
        return match_seasonal(dates=dates, seasonal=d).rename(name)

    n_doy = _help(nitrate_avg_seasonal(freq="D"), "nitrate_doy")
    n_woy = _help(nitrate_avg_seasonal(freq="W"), "nitrate_woy")
    n_moy = _help(nitrate_avg_seasonal(freq="M"), "nitrate_moy")
```

4. Combine everything and flatten out the buckets

```

    # merge all date-based features (EXCEPT weather, crucially), restricted to the
    # site's nitrate dates
    time_df = merge_on_dates(
        [n_daily, n_rolling, n_cal_D, n_cal_W, n_cal_M, pure_signal, n_doy, n_woy, n_moy],
        index=dates,
    )

    # attach the bucketed weather + crops/surplus by date (once per row, not per
    # bucket); inner join drops the tail nitrate dates that have no weather
    # (gridMET ends ~a month before the present, so the most recent days are cut).
    wide = flatten_buckets(cb, sb, wb)
    out = time_df.merge(wide, left_index=True, right_on="date", how="inner")

    out.insert(0, "site_uid", uid)  # (site_uid, date) is the row key
```

Here's an example of what changes when we change the buckets.

In [4]:
# hyperparameters
bucket1 = (50_000, 150_000)
bucket2 = (50_000, 150_000, 300_000)

buckets = [bucket1, bucket2]

small = "WQS0058"
med = "WQS0039"
large = "WQS0115"

def _helper(site, i):
    df = site_df(site, edges=buckets[i])
    print(f"Site {site}, bucket {i}:")
    print(f"  max_temp buckets: {df.filter(like="max_temp").columns}")
    
_helper(small, 0)
_helper(small, 1)
_helper(med, 0)
_helper(med, 1)
_helper(large, 0) 
_helper(large, 1)
    

Site WQS0058, bucket 0:
  max_temp buckets: Index(['max_temp_b0'], dtype='str')
Site WQS0058, bucket 1:
  max_temp buckets: Index(['max_temp_b0'], dtype='str')
Site WQS0039, bucket 0:
  max_temp buckets: Index(['max_temp_b0', 'max_temp_b1'], dtype='str')
Site WQS0039, bucket 1:
  max_temp buckets: Index(['max_temp_b0', 'max_temp_b1'], dtype='str')
Site WQS0115, bucket 0:
  max_temp buckets: Index(['max_temp_b0', 'max_temp_b1', 'max_temp_b2'], dtype='str')
Site WQS0115, bucket 1:
  max_temp buckets: Index(['max_temp_b0', 'max_temp_b1', 'max_temp_b2', 'max_temp_b3'], dtype='str')
